# California Birds — Colab Training

Runs v5 (ConvNeXt LR fix), Phase 2 (448px fine-tune), Phase 3 (EVA-02), and Phase 3b (EVA-02 continued) training on Colab GPU.

## Step 1: Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub timm pyyaml pillow

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q tensorboard

## Step 2: Download Dataset & Clone Repo

In [2]:
import kagglehub

# Download dataset (cached after first run)
dataset_path = kagglehub.dataset_download("anamethatiscreative/southern-california-birds")
print("Dataset path:", dataset_path)

# Check what's inside
import os
contents = os.listdir(dataset_path)
print("Contents:", contents)

# Find the actual data folder (with class subfolders)
# It may be dataset_path itself or a subfolder like dataset_path/data
if 'data' in contents:
    DATA_ROOT = os.path.join(dataset_path, 'data')
else:
    DATA_ROOT = dataset_path

num_classes = len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f"Data root: {DATA_ROOT}")
print(f"Number of classes: {num_classes}")

100%|██████████| 3.46G/3.46G [00:22<00:00, 167MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/anamethatiscreative/southern-california-birds/versions/5
Contents: ['data']
Data root: /root/.cache/kagglehub/datasets/anamethatiscreative/southern-california-birds/versions/5/data
Number of classes: 657


## Step 3: Clone repo & set paths

In [3]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/Model_California_Birds'
OUTPUT_DIR = '/content/drive/MyDrive/Model_California_Birds'

# Clone repo if not already present
if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/jihangli/Model_California_Birds.git {PROJECT_ROOT}
    
git checkout 

%cd {PROJECT_ROOT}

# Make sure output dir exists on Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Project: {PROJECT_ROOT}")
print(f"Data:    {DATA_ROOT}")
print(f"Output:  {OUTPUT_DIR}")

/content/drive/MyDrive/Model_California_Birds
Project: /content/drive/MyDrive/Model_California_Birds
Data:    /root/.cache/kagglehub/datasets/anamethatiscreative/southern-california-birds/versions/5/data
Output:  /content/drive/MyDrive/Model_California_Birds


In [ ]:
!git pull

remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 109 (delta 45), reused 103 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 361.02 KiB | 72.00 KiB/s, done.
Resolving deltas: 100% (45/45), completed with 8 local objects.
error: cannot lock ref 'refs/remotes/origin/jl_improve_model_03082026': Unable to create '/content/drive/MyDrive/Model_California_Birds/.git/refs/remotes/origin/jl_improve_model_03082026.lock': File exists.

Another git process seems to be running in this repository, e.g.
an editor opened by 'git commit'. Please make sure all processes
are terminated then try again. If it still fails, a git process
may have crashed in this repository earlier:
remove the file manually to continue.
From https://github.com/QAQWillQwQ/Model_California_Birds
 ! 86de02c..6f97cdb  jl_improve_model_03082026 -> origin/jl_improve_model_03082026  (unable to update local ref)


## Step 4: Check GPU

In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Torch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## Step 5: Create Colab config

Patches a config YAML with the correct Colab paths. Pick which run to do by changing `BASE_CONFIG`.

In [ ]:
import yaml

# ============================================================
# PICK YOUR RUN — uncomment ONE of these:
# ============================================================
# BASE_CONFIG = 'configs/convnext_base_v5_windows.yaml'              # Run 4: ConvNeXt v5 (LR fix)
# BASE_CONFIG = 'configs/convnext_base_finetune_448_windows.yaml'    # Run 5: Phase 2 (448px fine-tune)
# BASE_CONFIG = 'configs/eva02_base_windows.yaml'                    # Run 6: Phase 3 (EVA-02)
BASE_CONFIG = 'configs/eva02_base_finetune_windows.yaml'             # Run 7: Phase 3b (EVA-02 continued)
# ============================================================

COLAB_CONFIG = 'configs/colab_run.yaml'

with open(BASE_CONFIG, 'r') as f:
    cfg = yaml.safe_load(f)

# Patch paths for Colab
cfg['data_root'] = DATA_ROOT
cfg['output_dir'] = OUTPUT_DIR
cfg['num_workers'] = 4
cfg['prefetch_factor'] = 4
cfg['archive_outputs'] = False

# --- Colab overrides (adjust batch_size for your GPU) ---
# A100 80GB: batch_size 96 for 448px EVA-02
# T4 15GB:   batch_size 16 for 448px EVA-02
cfg['batch_size'] = 96

# For Phase 3b (EVA-02 continued): set resume_from to Run 6 best_full checkpoint
cfg['resume_from'] = '/content/drive/MyDrive/CS273P_Project/outputs/output_20260309_233227/checkpoints/eva02_base_patch14_448.mim_in22k_ft_in1k_best_full.pth'

# For Phase 2 (ConvNeXt 448): set resume_from to Run 4 best checkpoint (weights only)
# cfg['resume_from'] = '/content/drive/MyDrive/CS273P_Project/outputs/output_20260309_025748/checkpoints/convnext_base.fb_in22k_ft_in1k_best.pth'

with open(COLAB_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Created {COLAB_CONFIG} from {BASE_CONFIG}")
print(f"  model:         {cfg['model']}")
print(f"  image_size:    {cfg['image_size']}")
print(f"  batch_size:    {cfg['batch_size']}")
print(f"  learning_rate: {cfg['learning_rate']}")
print(f"  epochs:        {cfg['epochs']}")
print(f"  resume_from:   {cfg.get('resume_from', 'None')}")
print(f"  data_root:     {cfg['data_root']}")
print(f"  output_dir:    {cfg['output_dir']}")

Created configs/colab_run.yaml from configs/eva02_base_finetune_windows.yaml
  model:         eva02_base_patch14_448.mim_in22k_ft_in1k
  image_size:    448
  batch_size:    96
  learning_rate: 2e-05
  epochs:        60
  resume_from:   /content/drive/MyDrive/CS273P_Project/outputs/output_20260309_233227/checkpoints/eva02_base_patch14_448.mim_in22k_ft_in1k_best_full.pth
  data_root:     /kaggle/input/southern-california-birds/data
  output_dir:    /content/drive/MyDrive/Model_California_Birds


## Step 6: Run Training

In [5]:
# %load_ext tensorboard
!python src/train.py configs/yp_colab_run.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
  Val Batch [250/304] | Loss: 1.6494 | Top1: 0.8416 | Top5: 0.9511
  Val Batch [300/304] | Loss: 1.6433 | Top1: 0.8429 | Top5: 0.9517
  Val Batch [304/304] | Loss: 1.6443 | Top1: 0.8428 | Top5: 0.9515
  Val Epoch Time: 19.514s
Epoch [51/400] | Train Loss: 1.6714 | Train Top1: 0.5688 | Train Top5: 0.7263 | Val Loss: 1.6443 | Val Top1: 0.8428 | Val Top5: 0.9515 | LR: 1.93e-05 | Train Time: 311.950s | Val Time: 19.514s | Epoch Time: 331.464s

===== Epoch 52/400 (LR: 1.93e-05) =====
  Train Batch [50/1726] | Loss: 1.5605 | Top1: 0.5262 | Top5: 0.6500
  Train Batch [100/1726] | Loss: 1.6349 | Top1: 0.5609 | Top5: 0.7009
  Train Batch [150/1726] | Loss: 1.6828 | Top1: 0.5485 | Top5: 0.7063
  Train Batch [200/1726] | Loss: 1.6706 | Top1: 0.5592 | Top5: 0.7183
  Train Batch [250/1726] | Loss: 1.6500 | Top1: 0.5539 | Top5: 0.7094
  Train Batch [300/1726] | Loss: 1.6495 | Top1: 0.5553 | Top5: 0.7158
  Train Batch [350/1726] | Loss: 1.6178 | Top1: 0.5464 | Top5: 0.6994


In [ ]:
%tensorboard --logdir outputs
!tail -f train.log

## Step 7: Check Results

In [ ]:
import glob

# Find the latest output directory
output_dirs = sorted(glob.glob(f'{OUTPUT_DIR}/output_*'))
if output_dirs:
    latest = output_dirs[-1]
    print(f"Latest run: {latest}")
    !ls -la {latest}/checkpoints/
    print()
    !cat {latest}/logs/*_train_log.log | tail -20
else:
    print("No output directories found yet.")

Latest run: /content/drive/MyDrive/Model_California_Birds/output_20260310_103345_87dfc70a
ls: cannot access '/content/drive/MyDrive/Model_California_Birds/output_20260310_103345_87dfc70a/checkpoints/': No such file or directory

Epoch [14/30] | Train Loss: 0.1892 | Train Top1: 0.9466 | Train Top5: 0.9979 | Val Loss: 1.6870 | Val Top1: 0.6647 | Val Top5: 0.8693 | Train Time: 181.241s | Val Time: 7.207s | Epoch Time: 188.448s
Epoch [15/30] | Train Loss: 0.1738 | Train Top1: 0.9503 | Train Top5: 0.9981 | Val Loss: 1.6585 | Val Top1: 0.6821 | Val Top5: 0.8715 | Train Time: 180.975s | Val Time: 7.309s | Epoch Time: 188.284s
Epoch [16/30] | Train Loss: 0.1647 | Train Top1: 0.9529 | Train Top5: 0.9985 | Val Loss: 1.6948 | Val Top1: 0.6740 | Val Top5: 0.8679 | Train Time: 181.146s | Val Time: 7.394s | Epoch Time: 188.540s
Epoch [17/30] | Train Loss: 0.1549 | Train Top1: 0.9554 | Train Top5: 0.9988 | Val Loss: 1.6800 | Val Top1: 0.6794 | Val Top5: 0.8761 | Train Time: 181.160s | Val Time: 7.197